In [ ]:
# Data Wrangling
import pandas as pd
from pandas import Series, DataFrame
import numpy as np

# Visualization
import matplotlib.pylab as plt
from matplotlib import font_manager, rc
import seaborn as sns
%matplotlib inline
plt.rc('font',family='Malgun Gothic')
from matplotlib import font_manager, rc
# EDA
import klib

# Data Split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
seed = 42
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)

# Preprocessing & Feature Engineering
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PowerTransformer
from sklearn.feature_selection import SelectPercentile
from sklearn import model_selection

# Hyperparameter Optimization
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from bayes_opt import BayesianOptimization

# Modeling
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.base import ClassifierMixin

# Evaluation
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import log_loss
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Utility
import os
os.makedirs("submissions", exist_ok=True)
os.makedirs("models", exist_ok=True)
import time
import random
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image
import pickle
from tqdm import tqdm, tqdm_notebook
import platform
from itertools import combinations
from scipy.stats.mstats import gmean
import warnings
import tensorflow as tf
import kerastuner as kt
import optuna
from tensorflow import keras

In [ ]:
train = pd.read_csv('data/data_01_01.csv')
test = pd.read_csv('data/data_02_02.csv')
label = pd.read_csv('data/data_03_03.csv')

In [ ]:
q = test.cust.unique()

In [ ]:
train['y_train'] = train.cust.apply(lambda x:1 if x in q else 0)

In [ ]:
q = label.cust.unique()

In [ ]:
test['y_test'] = test.cust.apply(lambda x:1 if x in q else 0)

In [ ]:
set(train.columns)- set(test.columns)

In [ ]:
data_c = pd.concat([train, test]);data_c

In [ ]:
주구매 = data_c.filter(regex = '주구매상품대분류').columns.to_list()

In [ ]:
data_c.drop(주구매,axis = 1,inplace  = True)

In [ ]:
train = data_c.iloc[:len(train)]
test = data_c.iloc[len(train):]

In [ ]:
train.most_cop = train.most_cop.fillna('X')
test.most_cop = test.most_cop.fillna('X')

In [ ]:
train.most_chnl = train.most_chnl.fillna(0)
test.most_chnl = test.most_chnl.fillna(0)

train.point_count = train.point_count.fillna(train.point_count.mean())
test.point_count = test.point_count.fillna(test.point_count.mean())

train.point_sum = train.point_count.fillna(train.point_sum.mean())
test.point_sum = test.point_count.fillna(test.point_sum.mean())

In [ ]:
train = train.fillna(0)
test = test.fillna(0)

In [ ]:
y_train = train.y_train
del(train['y_train'])
y_test = test.y_test
del(test['y_test'])

In [ ]:
train_id=train.cust
del(train['cust'])
test_id=test.cust
del(test['cust'])

In [ ]:
del train["y_test"]
del test["y_train"]

In [ ]:
train=pd.get_dummies(data = train, columns = ['most_cop'], prefix = 'most_cop')
test=pd.get_dummies(data = test, columns = ['most_cop'], prefix = 'most_cop')

In [ ]:
X_train = train
X_test = test

del train
del test

In [ ]:
# standard Scaler 정규화 진행

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# 군집화 진행   # 5로 했을 때 성능이 좋아짐.  # 5, 10만 실험해봄,

from sklearn.cluster import KMeans
model = KMeans(n_clusters=5)
model.fit(X_train)

In [ ]:
X_train=pd.DataFrame(X_train)
X_test=pd.DataFrame(X_test)

In [ ]:
# 군집화 결과 피쳐 추가

X_train["group"]=model.predict(X_train)
X_test["group"]=model.predict(X_test)

- model 함수화

In [ ]:
# 시드 고정

def reset_seeds(reset_graph_with_backend=None):
    if reset_graph_with_backend is not None:
        K = reset_graph_with_backend
        K.clear_session()
        tf.compat.v1.reset_default_graph()
        print("KERAS AND TENSORFLOW GRAPHS RESET")  # optional

    np.random.seed(1)
    random.seed(2)
    tf.compat.v1.set_random_seed(3)
    os.environ['CUDA_VISIBLE_DEVICES'] = ''  # for GPU
    print("RANDOM SEEDS RESET")  # optional
   
reset_seeds()

In [ ]:
def lgbm_train(dic, lst, result):

    pbounds = {
        'n_estimators':(50,800),
        'learning_rate':(0.001,1.5),
        'max_depth':(2, 32),
        'num_leaves':(2, 64),
        'subsample':(0.5, 0.95),
        'colsample_bytree':(0.5, 0.95),
        'max_bin':(10, 500),
        'reg_lambda':(0.001, 50),
        'reg_alpha':(0.001, 50)
    }
    def lgbm_opt(n_estimators, learning_rate, max_depth, num_leaves,
                 subsample, colsample_bytree, max_bin, reg_lambda, reg_alpha):
        params = {
            "n_estimators":int(round(n_estimators)), 
            "learning_rate":learning_rate,
            'max_depth':int(round(max_depth)),
            'num_leaves':int(round(num_leaves)),
            'subsample':max(min(subsample, 1), 0),
            'colsample_bytree':max(min(colsample_bytree, 1), 0),
            'reg_lambda': reg_lambda,
            'reg_alpha': reg_alpha,
            'max_bin':int(max_bin)
        }


        lgbm = LGBMClassifier(objective= 'regression', metric= 'mse', random_state=0, **params, n_jobs=-1)
        lgbm.fit(X_train, y_train, early_stopping_rounds=30, eval_metric= ['logloss','accuracy'], eval_set=[(valid_X, valid_y)], verbose=False)
        score = cross_val_score(lgbm, X_train, y_train, scoring='neg_mean_absolute_error', cv=4, n_jobs=-1)

        return np.mean(score)

    BO_lgbm = BayesianOptimization(f = lgbm_opt, pbounds = pbounds, random_state=0)
    BO_lgbm.maximize(init_points=10, n_iter=10)

    max_params_lgbm = BO_lgbm.max['params']
    max_params_lgbm
    
    
    max_params_lgbm['n_estimators'] = int(max_params_lgbm['n_estimators'])
    max_params_lgbm['max_depth'] = int(max_params_lgbm['max_depth'])
    max_params_lgbm['max_bin'] = int(max_params_lgbm['max_bin'])
    max_params_lgbm['num_leaves'] = int(max_params_lgbm['num_leaves'])
    
    
    lgbm_clf = LGBMClassifier(objective= 'regression', metric= 'mse', random_state = 0, **max_params_lgbm)
    
    lgbm_clf.fit(X_train,y_train)
    y_pred = lgbm_clf.predict(X_test)
    y_pred_proba = lgbm_clf.predict_proba(X_test)[:,1]


    a=log_loss(y_test,y_pred_proba)
    b=roc_auc_score(y_test,y_pred_proba)
    c=accuracy_score(y_test,y_pred)
    
    result.append(y_pred_proba)
    dic["lgbm"]=lgbm_clf
    print("log_loss: ",a,"roc_auc : ", b, "accuracy : ",c)
    lst.append({"log_loss":a,"roc_auc":b,"accuracy":c})

In [ ]:
def dnn_train(dic, lst, result):
    
    def model_fn(hp):
        inputs = keras.Input(shape=(train_X.shape[1],))
        x = inputs
        for i in range(hp.Int('num_layers', 2, 3)):
            x = keras.layers.Dense(hp.Int('unit_'+str(i), 16, 64, step=16), activation='relu')(x)
            x = keras.layers.Dropout(hp.Float('dropout_'+str(i), 0, 0.5, step=0.25, default=0.5))(x)
        outputs = keras.layers.Dense(1, activation='sigmoid')(x)
        model = keras.Model(inputs, outputs)
        model.compile(loss='binary_crossentropy', 
                      optimizer=tf.keras.optimizers.Adam(hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])), 
                      metrics=[keras.metrics.AUC()])
        return model 
    
    
    tuner = kt.Hyperband(model_fn,
                         objective=kt.Objective('val_loss', direction="min"), 
                         max_epochs=5,
                         hyperband_iterations=2,
                         overwrite=True,
                         directory='dnn_tuning')

    tuner.search(train_X, train_y, validation_data=(valid_X, valid_y), 
                 callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)])

    model = tuner.get_best_models(1)[0]
    model.summary()
    
    y_pred = model.predict(X_test)
    #y_pred_proba = lgbm_clf.predict_proba(X_test)[:,1]
    
    #a=log_loss(y_test,y_pred)
    b=roc_auc_score(y_test,y_pred)
    #c=accuracy_score(y_test,y_pred)
    
    result.append(y_pred[:,0])
    dic["dnn"]=model
    print("log_loss: ","roc_auc : ", b, "accuracy : ")
    lst.append({"roc_auc":b})

In [ ]:
def lr_train(dic, lst, result):
    
    def lr_opt(trial):
        lr_penalty = trial.suggest_categorical("penalty", ["l1", "l2", "elasticnet", "none"])
        lr_c = trial.suggest_float("C", 0.01, 10, step = 0.05)

        lr_clf = LogisticRegression(
                penalty = lr_penalty,
                C = lr_c,
                n_jobs = -1,
                random_state = 44
        )

        score = np.mean(cross_val_score(lr_clf, X_train, y_train, scoring = "neg_mean_absolute_error", n_jobs = -1))
        return score

    study = optuna.create_study(direction="maximize")
    study.optimize(lr_opt, n_trials=20)
    
    lr_clf=LogisticRegression(**study.best_trial.params)
    lr_clf.fit(X_train,y_train)
    
    y_pred = lr_clf.predict(X_test)
    y_pred_proba = lr_clf.predict_proba(X_test)[:,1]


    a=log_loss(y_test,y_pred_proba)
    b=roc_auc_score(y_test,y_pred_proba)
    c=accuracy_score(y_test,y_pred)
    
    result.append(y_pred_proba)
    dic["lr"]=lr_clf
    print("log_loss: ",a,"roc_auc : ", b, "accuracy : ",c)
    lst.append({"log_loss":a,"roc_auc":b,"accuracy":c})

In [ ]:
def ada_train(dic, lst, result):
    pbounds = { 'learning_rate': (0.01, 1.5),
            'n_estimators': (50, 100)}

    def ada_opt(learning_rate, n_estimators):

        params = {
            'learning_rate': learning_rate,
            'n_estimators' : int(round(n_estimators))
        }

        ada = AdaBoostClassifier(**params,random_state=0,)
        score = cross_val_score(ada, X_train, y_train, scoring='neg_mean_absolute_error', cv=4, n_jobs=-1)
        return np.mean(score)

    BO_ada = BayesianOptimization(f = ada_opt, pbounds = pbounds, random_state=0)
    BO_ada.maximize(init_points=10, n_iter=10)

    max_params_ada = BO_ada.max['params']
    max_params_ada
    
    max_params_ada['n_estimators'] = int(max_params_ada['n_estimators'])
    ada_clf = AdaBoostClassifier(random_state = 0, **max_params_ada)
    ada_clf.fit(X_train,y_train)

    y_pred = ada_clf.predict(X_test)
    y_pred_proba = ada_clf.predict_proba(X_test)[:,1]


    a=log_loss(y_test,y_pred_proba)
    b=roc_auc_score(y_test,y_pred_proba)
    c=accuracy_score(y_test,y_pred)
    
    result.append(y_pred_proba)
    dic["ada"]=ada_clf
    print("log_loss: ",a,"roc_auc : ", b, "accuracy : ",c)
    lst.append({"log_loss":a,"roc_auc":b,"accuracy":c})


In [ ]:
def cat_train(dic, lst, result):
    c_model = CatBoostClassifier(random_state = 44)
    c_model.fit(X_train,y_train)
    
    y_pred = c_model.predict(X_test)
    y_pred_proba = c_model.predict_proba(X_test)[:,1]


    a=log_loss(y_test,y_pred_proba)
    b=roc_auc_score(y_test,y_pred_proba)
    c=accuracy_score(y_test,y_pred)
    
    
    result.append(y_pred_proba)
    dic["cat"]=c_model
    print("log_loss: ",a,"roc_auc : ", b, "accuracy : ",c)
    lst.append({"log_loss":a,"roc_auc":b,"accuracy":c})

In [ ]:
def rf_train(dic, lst, result):
    pbounds = { 'n_estimators': (50,250),
            'max_depth': (5,15), 
            'max_features': (0.8,0.95),
            'min_samples_leaf': (1, 5)}

    def rf_opt(n_estimators, max_depth, max_features, min_samples_leaf):
        params = {'n_estimators' : int(round(n_estimators)),
            'max_depth' : int(round(max_depth)),
            'min_samples_leaf' : int(round(min_samples_leaf))}

        rf = RandomForestClassifier(**params, n_jobs = -1, random_state = 0)
        score = cross_val_score(rf, X_train, y_train, scoring = 'roc_auc', cv = 4, n_jobs = -1)
        return np.mean(score)


    BO_rf = BayesianOptimization(f = rf_opt, pbounds = pbounds, random_state=0)
    BO_rf.maximize(init_points=5, n_iter=5)

    max_params_rf = BO_rf.max['params']
    
    max_params_rf['n_estimators'] = int(max_params_rf['n_estimators'])
    max_params_rf['max_depth'] = int(max_params_rf['max_depth'])
    max_params_rf['min_samples_leaf'] = int(max_params_rf['min_samples_leaf'])
    rf_clf = RandomForestClassifier(random_state = 0, **max_params_rf)
    rf_clf.fit(X_train,y_train)
    
    y_pred = rf_clf.predict(X_test)
    y_pred_proba = rf_clf.predict_proba(X_test)[:,1]


    a=log_loss(y_test,y_pred_proba)
    b=roc_auc_score(y_test,y_pred_proba)
    c=accuracy_score(y_test,y_pred)
    
    
    result.append(y_pred_proba)
    dic["rf"]=rf_clf
    print("log_loss: ",a,"roc_auc : ", b, "accuracy : ",c)
    lst.append({"log_loss":a,"roc_auc":b,"accuracy":c})

In [ ]:
model_dict={}; model_dict1={} ; model_dict2={}   # 학습한 모델 저장
score_lst=[]; score_lst1=[]; score_lst2=[]  # validation 점수 저장   딕셔너리 형태 key 값 : log_loss, roc_auc, accuracy
result_lst=[]; result_lst1=[]; result_lst2=[]


lst_e=[[model_dict, score_lst, result_lst], [model_dict1, score_lst1, result_lst1], [model_dict2, score_lst2, result_lst2]]

t = X_train
k = X_test
a_1=random.sample(list(t.columns),int(len(list(t.columns))//2))
b_1=random.sample(list(t.columns),int(len(list(t.columns))//2))
c_1=random.sample(list(t.columns),int(len(list(t.columns))//2))

column_lst=[a_1, b_1, c_1]

# 주의 오래걸림! 모든 모델 훈련 18(3*6)개의 모델 튜닝 및 훈련
def ensemble():
    global train_X, valid_X, train_y, valid_y, X_train, y_train, X_test
    for z,i in tqdm(enumerate(lst_e)):
        X_train=t[column_lst[z]]
        X_test=k[column_lst[z]]
        train_X, valid_X, train_y, valid_y = train_test_split(X_train, y_train, test_size=0.3, shuffle=False, random_state=34)

        def train():
            lr_train(i[0], i[1], i[2]); print("LR FINISH")
            lgbm_train(i[0], i[1], i[2]); print("LGBM FINISH")
            dnn_train(i[0], i[1], i[2]); print("DNN FINISH")
            ada_train(i[0], i[1], i[2]); print("ADA FINISH")
            cat_train(i[0], i[1], i[2]); print("CAT FINISH")
            rf_train(i[0], i[1], i[2]); print("RF FINISH")
        train()
        print("Finish", z+1, "번째")
        
ensemble()

In [ ]:
# Y_test에 대한 submission 결합.
def submission():
    global df
    df=pd.DataFrame(test_id)
    for l,model_dict in enumerate(lst_e):
        for p,model in enumerate(model_dict[0]):
            df=pd.concat([df,pd.DataFrame({str(model):model_dict[2][p]})],axis=1)
    df=df.set_index("cust")
submission()

In [ ]:
# 모형의 예측값 간의 상관관계를 보기 위해 hitmap을 도식한다.
plt.figure(figsize = (16,9))
g = sns.heatmap(df.corr(), annot=True, cmap='YlGnBu')
g.set_title("Correlation between ensembles of models")
plt.show()

In [ ]:
for a in lst_e:
    print(a[1], end='\n'*3)

In [ ]:
# 데이터프레임에 따른 모델 컬럼명 구분을 위한 컬럼명 변경

df.columns=["lr", "lgbm", "dnn", "ada", "cat", "rf", "lr1", "lgbm1", "dnn1", "ada1", "cat1", "rf1", "lr2", "lgbm2", "dnn2", "ada2", "cat2", "rf2"]

In [ ]:
# 서브미션 앙상블

def ensemble_submission(submissions,p,power=None,name=""):
    if p==0 and power==[]:
        pred=gmean(submissions,axis=1)
        
    elif p!=0 and power==[]:
        pred = (np.sum(np.array(submissions)**p, axis=1) / len(submissions.columns))**(1/p)
    
    else: 
        for i,a in enumerate(power):
            submissions.iloc[:,i]=submissions.iloc[:,i]*a
        pred=np.sum(np.array(submissions),axis=1)/sum(power)
    pd.DataFrame({"ID":test_id,"STATUS":pred}).to_csv("submissions/"+name+".csv",index=False)
    ok= pd.DataFrame({"ID":test_id,"STATUS":pred})
    ok["STATUS_1"]=ok["STATUS"].apply(lambda x: 1 if x>=0.5 else 0)
    
    return accuracy_score(y_test,ok.iloc[:,2]), log_loss(y_test,ok.iloc[:,1])

In [ ]:
# 서브미션 콤비네이션 확인하여 loss가 가장 작은 앙상블 찾기.

max_score = 100
for p in tqdm([0, 1, 1.5]):  # p==1:산술평균, p=0:기하평균, 그 외:멱평균(주의:멱평균은 과적합 가능성이 높음)    
    for i in range(2, 5):
        for models in list(combinations(df.columns, i)):
            score=ensemble_submission(df[list(models)], p, power=[], name=str(p))
            if max_score > score[1]:
                max_score = score[1]
                sc=score
                best=models
                print(score, models, p)
        print(sc, best, p)
    print(sc, best, p)

In [ ]:
# 수동 서브미션 앙상블 지정.

data=df.loc[:,list(best)]

In [ ]:
def ensemble_submission(submissions,p,power=None,name=""):
    if p==0 and power==[]:
        pred=gmean(submissions,axis=1)
        
    elif p!=0 and power==[]:
        pred = (np.sum(np.array(submissions)**p, axis=1) / len(submissions.columns))**(1/p)
    
    else: 
        for i,a in enumerate(power):
            submissions.iloc[:,i]=submissions.iloc[:,i]*a
        pred=np.sum(np.array(submissions),axis=1)/sum(power)
    pd.DataFrame({"ID":test_id,"STATUS":pred}).to_csv("submissions/"+name+".csv",index=False)
    return pd.DataFrame({"ID":test_id,"STATUS":pred})

In [ ]:
ok=ensemble_submission(data, 1.5, power=[], name="###")

In [ ]:
ok

In [ ]:
ok["STATUS_2"]= ok["STATUS"].apply(lambda x: np.round(x,2))

plt.figure(figsize=(8,4))
ax=sns.countplot(x="STATUS_2", data=ok)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
plt.tight_layout()
plt.title("모델이 예측한 다음달 고객이 올 확률 count")
plt.show()

In [ ]:
import joblib
for i,mo in enumerate(lst_e):
    for m in mo[0]:
        if m=="dnn":
            mo[0][m].save("models/"+str(i)+"_"+m)
        else:
            joblib.dump(mo[0][m], "models/"+str(i)+"_"+m+".pkl")